# CRC methylation panel: selection and validation

Runs the full analysis of *Genome-Wide Methylation Profiling Uncovers Specific Biomarkers for Colon Cancer Diagnosis* from public data (TCGA via the GDC Xena hub; GEO GSE48684, GSE131013, GSE40279, GSE35069 and GSE122126).

Runs on the free CPU runtime (~12 GB RAM). Every stage caches to `WORK`, so after a disconnect re-run the same cell: finished stages are skipped. The full run takes several hours, mostly downloading and streaming the 11 TCGA methylation matrices.

In [ ]:
# Package: cloned from GitHub, or taken from crc_meth_repro.tar.gz uploaded to /content.
import os, subprocess, sys
PKG = '/content/crc_meth_repro'
REPO = 'https://github.com/masoudrezaei082-afk/crc-methylation-panel.git'
if not os.path.isdir(PKG) and os.path.exists('/content/crc_meth_repro.tar.gz'):
    subprocess.run(['tar', 'xzf', '/content/crc_meth_repro.tar.gz', '-C', '/content'], check=True)
if not os.path.isdir(PKG):
    subprocess.run(['git', 'clone', '--depth', '1', REPO, PKG], check=True)
assert os.path.isdir(PKG), 'package not found'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', f'{PKG}/requirements.txt'], check=True)
WORK, RES = '/content/work', '/content/results'

In [ ]:
# Unit tests (synthetic data, no internet access)
!cd {PKG} && python -m pytest -q tests

In [ ]:
# All stages: annot coad pancan dmc rna integrate specificity network geo classify blood report
# Restrict with --stage NAME, or --from / --until; --force recomputes.
!cd {PKG} && python run_all.py --workdir {WORK} --resultsdir {RES} 2>&1 | tee -a {RES}/logs_run.txt

In [ ]:
import pandas as pd
from IPython.display import Markdown, Image, display
display(pd.read_csv(f'{RES}/tables/traceability_network_prioritized_panel.csv', index_col=0))
display(pd.read_csv(f'{RES}/tables/classification_external.csv'))

In [ ]:
import glob
for f in sorted(glob.glob(f'{RES}/figures/*.png')):
    print(f); display(Image(f))

In [ ]:
!cd /content && tar czf results.tar.gz results && ls -lh results.tar.gz